# Cartographie spatiale des résultats côtiers

**GGOSSS 2026 · Jour 2 · Instructeur: Nourdi Njutapvoui**

Ce notebook complète les deux TP. Il produit des cartes spatiales lisibles pour discuter les résultats: facteurs PAK/Kribi, scénarios de priorité, prédictions des algorithmes de machine learning et localisation des erreurs.

## Objectifs pédagogiques

- Situer les zones d'étude dans le Golfe de Guinée avec un vrai fond continental.
- Cartographier les cinq facteurs PAK/Kribi: Shoreline, SWV, marée, vagues et vent.
- Comparer spatialement les prédictions des six algorithmes ML.
- Identifier où les modèles se trompent, pas seulement combien ils se trompent.
- Relier les cartes à une décision: suivi terrain, validation indépendante et priorisation.

In [ ]:
# Bloc de préparation: importer les bibliothèques, définir les chemins et fixer le style graphique.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_DISPONIBLE = True
except Exception:
    CARTOPY_DISPONIBLE = False

COULEURS_GGOSSS = {
    "bleu": "#17304f",
    "ocean": "#0077b6",
    "cyan": "#00a6a6",
    "sable": "#f2c14e",
    "corail": "#f05d5e",
    "gris": "#607d8b",
}

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "axes.facecolor": "#fbfcfd",
    "axes.edgecolor": COULEURS_GGOSSS["bleu"],
    "axes.titleweight": "bold",
})

def trouver_dossier_donnees():
    """Trouver le dossier de données dans l'organisation GitHub ou dans l'organisation locale instructeur."""
    for candidat in [Path("../data"), Path("../Datasets"), Path("data"), Path("Datasets")]:
        if candidat.exists():
            return candidat
    raise FileNotFoundError("Dossier de donnees introuvable. Lancer le notebook depuis le dossier notebooks ou la session.")

DOSSIER_DONNEES = trouver_dossier_donnees()
DOSSIER_DONNEES

In [ ]:
# Bloc cartographique: fonction utilitaire pour dessiner un fond continental réaliste avec Cartopy.
def ajouter_fond_carte(ax, emprise, titre):
    """Ajouter continent, océan, trait de côte, frontières et grille géographique."""
    if CARTOPY_DISPONIBLE:
        ax.set_extent(emprise, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor="#f1efe6")
        ax.add_feature(cfeature.OCEAN, facecolor="#d8eef7")
        ax.add_feature(cfeature.COASTLINE, linewidth=0.9)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        grille = ax.gridlines(draw_labels=True, linewidth=0.25, color="#6b7280", alpha=0.5)
        grille.top_labels = False
        grille.right_labels = False
    else:
        ax.set_xlim(emprise[0], emprise[1])
        ax.set_ylim(emprise[2], emprise[3])
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_facecolor("#d8eef7")
    ax.text(
        0.03, 0.95, titre,
        transform=ax.transAxes,
        fontsize=9,
        weight="bold",
        va="top",
        bbox={"facecolor": "white", "alpha": 0.82, "edgecolor": "none", "pad": 2},
    )

## Carte 1 - Facteurs PAK/Kribi

Cette carte permet de comparer les cinq facteurs et l'indice composite. Elle sert à poser une question simple aux participants: **le secteur prioritaire reste-t-il le même selon le facteur observé?**

In [ ]:
# Bloc PAK: charger les facteurs et les localités, puis construire une planche de cartes spatiales.
facteurs = pd.read_csv(DOSSIER_DONNEES / "pak_integrated_coastal_factors.csv")
localites = pd.read_csv(DOSSIER_DONNEES / "selected_localities_for_maps.csv")

colonnes_facteurs = [
    ("shoreline_erosion_score", "Shoreline"),
    ("swv_score", "SWV"),
    ("tide_score", "Marée"),
    ("wave_score", "Vagues"),
    ("wind_score", "Vent"),
    ("coastal_dynamics_pressure_index", "Indice composite"),
]

projection = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
fig = plt.figure(figsize=(14, 8), constrained_layout=True)
emprise_pak = [9.55, 10.15, 2.15, 3.35]

for i, (colonne, titre) in enumerate(colonnes_facteurs, start=1):
    ax = fig.add_subplot(2, 3, i, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(2, 3, i)
    ajouter_fond_carte(ax, emprise_pak, f"PAK/Kribi - {titre}")
    transform = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
    kwargs = {"transform": transform} if CARTOPY_DISPONIBLE else {}
    ax.scatter(localites["Lon"], localites["Lat"], marker="^", s=34, color=COULEURS_GGOSSS["bleu"], zorder=4, **kwargs)
    for _, ligne in localites[localites["Name"].isin(["Lokoundje", "Londji", "Kribi", "Campo", "Ebodje"])].iterrows():
        ax.text(ligne["Lon"] + 0.006, ligne["Lat"] + 0.006, ligne["Name"], fontsize=6.5, **kwargs)
    points = ax.scatter(
        facteurs["Longitude"], facteurs["Latitude"], c=facteurs[colonne], s=185,
        cmap="YlOrRd", vmin=0, vmax=1, edgecolors="none", linewidths=0, zorder=5, **kwargs
    )
    for _, ligne in facteurs.iterrows():
        ax.text(ligne["Longitude"] + 0.01, ligne["Latitude"] + 0.01, ligne["Point"], fontsize=8, weight="bold", **kwargs)

barre = fig.colorbar(points, ax=fig.axes, shrink=0.78, pad=0.03)
barre.set_label("Score normalisé / indice")
fig.suptitle("GGOSSS 2026 - Cartes spatiales des facteurs PAK/Kribi", fontsize=14, weight="bold")
plt.show()

## Carte 2 - Résultats spatiaux des algorithmes ML

Cette carte transforme la comparaison des modèles en discussion spatiale. Deux questions guident l'analyse: **où les modèles prédisent-ils les classes fortes?** et **où se concentrent les erreurs?**

In [ ]:
# Bloc ML: charger ou créer les prédictions spatiales des six algorithmes.
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

def trouver_fichier_ml(nom_fichier):
    """Trouver un fichier ML dans le dossier data de la session GitHub."""
    return DOSSIER_DONNEES / nom_fichier

chemin_predictions = trouver_fichier_ml("cameroon_ml_predictions_spatiales_sample.csv")

if chemin_predictions.exists():
    predictions = pd.read_csv(chemin_predictions)
else:
    df = pd.read_csv(trouver_fichier_ml("cameroon_coastal_vulnerability_ml.csv"))
    variables_base = [
        "coastal_slope", "elevation_m", "shoreline_change_myr", "sea_level_anomaly_m",
        "tide_m", "significant_wave_height_m", "land_surface_temperature", "wind_speed_ms",
    ]
    cible = df["vulnerability_class"]
    masque_entrainement = df["x_utm"] <= df["x_utm"].median()
    transformateur = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)
    lon, lat = transformateur.transform(df["x_utm"].to_numpy(), df["y_utm"].to_numpy())
    predictions = df[["x_utm", "y_utm", "vulnerability_class"]].copy()
    predictions["longitude"] = lon
    predictions["latitude"] = lat
    predictions["validation_spatiale"] = np.where(masque_entrainement, "entrainement", "test")
    modeles = {
        "SVM": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale")),
        "RF": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=80, random_state=42, class_weight="balanced")),
        "ANN_MLP": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(30, 15), max_iter=400, random_state=42, early_stopping=True)),
        "DT": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=8, random_state=42, class_weight="balanced")),
        "LR": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
        "KNN": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    }
    for nom, modele in modeles.items():
        modele.fit(df.loc[masque_entrainement, variables_base], cible.loc[masque_entrainement])
        prediction = modele.predict(df[variables_base])
        predictions[f"prediction_{nom}"] = prediction
        predictions[f"erreur_{nom}"] = prediction != cible
    predictions = predictions.sample(n=min(3500, len(predictions)), random_state=42)

predictions.head()

In [ ]:
# Bloc de cartes ML: représenter les classes prédites par chacun des six algorithmes.
from matplotlib.colors import BoundaryNorm, ListedColormap

modeles = ["SVM", "RF", "ANN_MLP", "DT", "LR", "KNN"]
carte_couleurs_ivci = ListedColormap(["#1a9850", "#91cf60", "#ffff66", "#fdae61", "#d7191c"])
normalisation_ivci = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], carte_couleurs_ivci.N)
emprise_ml = [
    predictions["longitude"].min() - 0.2,
    predictions["longitude"].max() + 0.2,
    predictions["latitude"].min() - 0.2,
    predictions["latitude"].max() + 0.2,
]
localites_cameroun = pd.DataFrame([
    {"Name": "Limbe", "Lon": 9.21, "Lat": 4.02},
    {"Name": "Douala", "Lon": 9.70, "Lat": 4.05},
    {"Name": "Kribi", "Lon": 9.91, "Lat": 2.94},
    {"Name": "Campo", "Lon": 9.82, "Lat": 2.38},
])

projection = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
fig = plt.figure(figsize=(14, 8), constrained_layout=True)
for i, modele in enumerate(modeles, start=1):
    ax = fig.add_subplot(2, 3, i, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(2, 3, i)
    ajouter_fond_carte(ax, emprise_ml, f"Prédiction {modele}")
    transform = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
    kwargs = {"transform": transform} if CARTOPY_DISPONIBLE else {}
    points = ax.scatter(
        predictions["longitude"], predictions["latitude"], c=predictions[f"prediction_{modele}"],
        s=16, cmap=carte_couleurs_ivci, norm=normalisation_ivci, alpha=0.88, edgecolors="none", linewidths=0, zorder=4, **kwargs
    )
    lon_split = predictions["longitude"].median()
    ax.plot([lon_split, lon_split], [emprise_ml[2], emprise_ml[3]], linestyle="--", color=COULEURS_GGOSSS["bleu"], linewidth=1, **kwargs)
    ax.scatter(localites_cameroun["Lon"], localites_cameroun["Lat"], marker="^", s=34, color=COULEURS_GGOSSS["bleu"], zorder=6, **kwargs)
    for _, ligne in localites_cameroun.iterrows():
        ax.text(ligne["Lon"] + 0.015, ligne["Lat"] + 0.015, ligne["Name"], fontsize=6.8, **kwargs)

barre = fig.colorbar(points, ax=fig.axes, shrink=0.78, pad=0.03)
barre.set_ticks([1, 2, 3, 4, 5])
barre.set_label("Classe de vulnérabilité prédite")
fig.suptitle("GGOSSS 2026 - Représentation spatiale des prédictions ML", fontsize=14, weight="bold")
plt.show()

In [ ]:
# Bloc d'erreurs ML: localiser les zones où chaque algorithme se trompe.
fig = plt.figure(figsize=(14, 8), constrained_layout=True)
for i, modele in enumerate(modeles, start=1):
    ax = fig.add_subplot(2, 3, i, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(2, 3, i)
    ajouter_fond_carte(ax, emprise_ml, f"Erreurs {modele}")
    transform = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
    kwargs = {"transform": transform} if CARTOPY_DISPONIBLE else {}
    ok = predictions[~predictions[f"erreur_{modele}"]]
    erreurs = predictions[predictions[f"erreur_{modele}"]]
    ax.scatter(ok["longitude"], ok["latitude"], s=11, color="#9ca3af", alpha=0.35, edgecolors="none", linewidths=0, zorder=3, **kwargs)
    ax.scatter(erreurs["longitude"], erreurs["latitude"], s=24, color="#d7191c", alpha=0.85, edgecolors="none", linewidths=0, zorder=4, **kwargs)
    lon_split = predictions["longitude"].median()
    ax.plot([lon_split, lon_split], [emprise_ml[2], emprise_ml[3]], linestyle="--", color=COULEURS_GGOSSS["bleu"], linewidth=1, **kwargs)
    ax.text(0.03, 0.05, f"{len(erreurs)} erreurs / {len(predictions)} points", transform=ax.transAxes, fontsize=8, bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"})

fig.suptitle("GGOSSS 2026 - Localisation des erreurs de prédiction ML", fontsize=14, weight="bold")
plt.show()

## Cartes interactives en ligne

Les cartes HTML ci-dessous utilisent OpenStreetMap et Esri. Elles sont utiles en présentation car les couches peuvent être activées/désactivées.

- `../Shared_assets/interactive_maps/ggosss2026_pak_kribi_carte_interactive.html`
- `../Shared_assets/interactive_maps/ggosss2026_ml_cartes_algorithmes.html`

Pour le dépôt GitHub officiel, garder seulement les petits CSV et notebooks nécessaires. Les HTML interactifs et les PNG lourds peuvent rester dans le dossier instructeur local.

## Carte 4 - Synthèse de décision

Cette dernière carte sert à conclure le TP. Elle ne remplace pas l'analyse détaillée: elle force les participants à transformer plusieurs résultats en recommandation argumentée.

- À gauche: priorité PAK/Kribi issue des cinq facteurs.
- À droite: priorité ML issue de la classe moyenne prédite, du désaccord entre algorithmes et de la fréquence d'erreur.
- La taille des points ML augmente lorsque les modèles sont moins d'accord.


In [ ]:
# Bloc synthèse décisionnelle: combiner les facteurs PAK/Kribi et l'incertitude ML dans une seule figure.
from matplotlib.colors import BoundaryNorm, ListedColormap

facteurs_decision = pd.read_csv(DOSSIER_DONNEES / "pak_priorites_decision_suivi.csv")
predictions_decision = pd.read_csv(trouver_fichier_ml("cameroon_ml_predictions_spatiales_sample.csv"))

colonnes_modeles = ["prediction_SVM", "prediction_RF", "prediction_ANN_MLP", "prediction_DT", "prediction_LR", "prediction_KNN"]
colonnes_erreurs = ["erreur_SVM", "erreur_RF", "erreur_ANN_MLP", "erreur_DT", "erreur_LR", "erreur_KNN"]
predictions_decision["classe_moyenne_ml"] = predictions_decision[colonnes_modeles].mean(axis=1)
predictions_decision["desaccord_ml"] = predictions_decision[colonnes_modeles].std(axis=1)
predictions_decision["frequence_erreur"] = predictions_decision[colonnes_erreurs].mean(axis=1)
predictions_decision["score_decision_ml"] = (
    (predictions_decision["classe_moyenne_ml"] - 1) / 4 * 0.55
    + predictions_decision["desaccord_ml"].clip(0, 1.7) / 1.7 * 0.25
    + predictions_decision["frequence_erreur"] * 0.20
)
predictions_decision["priorite_ml"] = pd.cut(
    predictions_decision["score_decision_ml"],
    bins=[-0.01, 0.33, 0.55, 1.01],
    labels=["faible", "moyenne", "haute"],
)

couleurs_priorite = {"faible": "#2a9d8f", "moyenne": "#f2c14e", "haute": "#d7191c"}
carte_priorite = ListedColormap([couleurs_priorite["faible"], couleurs_priorite["moyenne"], couleurs_priorite["haute"]])
norme_priorite = BoundaryNorm([0, 1, 2, 3], carte_priorite.N)
codes_priorite_ml = predictions_decision["priorite_ml"].map({"faible": 0.5, "moyenne": 1.5, "haute": 2.5}).astype(float)

fig = plt.figure(figsize=(14, 7), constrained_layout=True)
ax_pak = fig.add_subplot(1, 2, 1, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(1, 2, 1)
ax_ml = fig.add_subplot(1, 2, 2, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(1, 2, 2)
transform = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
kwargs = {"transform": transform} if CARTOPY_DISPONIBLE else {}

ajouter_fond_carte(ax_pak, [9.55, 10.15, 2.15, 3.35], "Décision PAK/Kribi")
for _, ligne in facteurs_decision.iterrows():
    couleur = couleurs_priorite[str(ligne["priorite_decision"])]
    ax_pak.scatter(ligne["Longitude"], ligne["Latitude"], s=300, color=couleur, edgecolors="none", alpha=0.92, zorder=5, **kwargs)
    ax_pak.text(ligne["Longitude"] + 0.012, ligne["Latitude"] + 0.012, f'{ligne["Point"]} | rang {ligne["rang"]}', fontsize=8.5, weight="bold", **kwargs)

emprise_ml = [
    predictions_decision["longitude"].min() - 0.22,
    predictions_decision["longitude"].max() + 0.22,
    predictions_decision["latitude"].min() - 0.22,
    predictions_decision["latitude"].max() + 0.22,
]
ajouter_fond_carte(ax_ml, emprise_ml, "Décision ML")
ax_ml.scatter(
    predictions_decision["longitude"],
    predictions_decision["latitude"],
    c=codes_priorite_ml,
    cmap=carte_priorite,
    norm=norme_priorite,
    s=20 + predictions_decision["desaccord_ml"].fillna(0).to_numpy() * 18,
    alpha=0.86,
    edgecolors="none",
    linewidths=0,
    zorder=5,
    **kwargs,
)
ligne_split = predictions_decision["longitude"].median()
ax_ml.plot([ligne_split, ligne_split], [emprise_ml[2], emprise_ml[3]], linestyle="--", color=COULEURS_GGOSSS["bleu"], linewidth=1.2, **kwargs)

legende = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=couleur, markersize=9, label=f"priorité {nom}")
    for nom, couleur in couleurs_priorite.items()
]
ax_pak.legend(handles=legende, loc="lower left", fontsize=8, title="Lecture décisionnelle")
ax_ml.legend(handles=legende, loc="lower left", fontsize=8, title="Lecture décisionnelle")
fig.suptitle("GGOSSS 2026 - Carte synthèse pour prioriser le suivi côtier", fontsize=14, weight="bold")
plt.show()

## Figure de synthese - typologie workshop par zone

Cette section reprend la logique visuelle des figures de typologie par zone, mais elle est adaptee aux resultats du workshop. Les familles de processus sont calculees a partir des cinq facteurs utilises dans le TP.

In [ ]:
# Bloc typologie: charger la synthese par zone et verifier les proportions des familles de processus.
typologie = pd.read_csv(DOSSIER_DONNEES / 'pak_typologie_processus_workshop_par_zone.csv')
colonnes_processus = ['Shoreline_pct', 'ForcageMarin_pct', 'Vent_pct']
typologie['somme_pct'] = typologie[colonnes_processus].sum(axis=1).round(1)
display(typologie[['Point', 'Zone', 'coastal_dynamics_pressure_index', *colonnes_processus, 'FamilleDominante', 'somme_pct']])

In [ ]:
# Bloc barres empilees: visualiser la contribution relative des familles de processus par zone.
fig, ax = plt.subplots(figsize=(9, 4.8))
couleurs = {'Shoreline_pct': '#d7301f', 'ForcageMarin_pct': '#2166ac', 'Vent_pct': '#31a354'}
labels = {'Shoreline_pct': 'Shoreline', 'ForcageMarin_pct': 'Forcage marin', 'Vent_pct': 'Vent'}
gauche = np.zeros(len(typologie))
for col in colonnes_processus:
    ax.barh(typologie['Zone'], typologie[col], left=gauche, color=couleurs[col], edgecolor='white', label=labels[col])
    gauche += typologie[col].to_numpy()
ax.set_xlim(0, 100)
ax.set_xlabel('Contribution relative (%)')
ax.set_ylabel('Zone pedagogique')
ax.set_title('Typologie workshop des processus cotiers par zone')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Bloc correlation: explorer les relations entre les cinq facteurs avant la discussion de decision.
facteurs = ['shoreline_erosion_score', 'swv_score', 'tide_score', 'wave_score', 'wind_score']
corr = typologie[facteurs].corr()
fig, ax = plt.subplots(figsize=(6.3, 5.3))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(facteurs)), facteurs, rotation=35, ha='right')
ax.set_yticks(range(len(facteurs)), facteurs)
for i in range(len(facteurs)):
    for j in range(len(facteurs)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Correlation exploratoire entre facteurs du workshop')
plt.tight_layout()
plt.show()